<a href="https://colab.research.google.com/github/mariborges22/Supply-Chain-Prevision-with-Xgboost/blob/nome-da-sua-nova-branch/Previs%C3%A3o_de_estoques_com_XGBOOST_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### 1.1 Montagem do Google Drive

O primeiro passo na execução deste notebook é montar o seu Google Drive. Isso é necessário para que o notebook possa acessar e ler o arquivo do dataset (`bi_movimentacao.xlsx`) que está armazenado no seu Drive. A célula de código abaixo realiza essa montagem, tornando o conteúdo do seu Google Drive acessível no caminho `/content/drive/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#### 1.3 Importação de Bibliotecas

Para realizar o pré-processamento dos dados, a construção do modelo de Machine Learning e a análise dos resultados, diversas bibliotecas Python são necessárias. A célula abaixo importa as bibliotecas que serão utilizadas ao longo do notebook, juntamente com uma breve explicação de suas finalidades:

*   **`pandas`**: Essencial para manipulação e análise de dados, especialmente com DataFrames. Utilizada para carregar, limpar, transformar e agregar o dataset.
*   **`numpy`**: Oferece suporte a arrays e matrizes, além de funções matemáticas de alto nível. Frequentemente utilizada em operações numéricas e em conjunto com pandas e bibliotecas de machine learning.
*   **`matplotlib.pyplot`**: Uma biblioteca para a criação de visualizações estáticas, interativas e animadas em Python. Usada para gerar gráficos e plots.
*   **`seaborn`**: Baseada em matplotlib, fornece uma interface de alto nível para desenhar gráficos estatísticos atraentes e informativos. Utilizada para visualizações mais complexas, como boxplots e distribuições.
*   **`warnings.filterwarnings`**: Utilizada para gerenciar avisos que podem ser gerados durante a execução do código, ajudando a manter a saída do notebook mais limpa.
*   **`sklearn.ensemble.RandomForestRegressor`**: Embora o foco principal seja XGBoost, RandomForestRegressor é um algoritmo de ensemble baseado em árvores que pode ser utilizado para tarefas de regressão e comparação de modelos (note: no fluxo atual, o foco mudou para XGBoost agregado, mas a importação pode ter sido de um fluxo anterior ou para experimentação).
*   **`time`**: Fornece funções relacionadas a tempo. Pode ser usado para medir o tempo de execução de blocos de código.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from warnings import filterwarnings
from sklearn.ensemble import RandomForestRegressor
import time

## 2. Pré-processamento dos Dados

O objetivo desta etapa é preparar o dataset para o treinamento do modelo de Machine Learning. Isso envolve limpar os dados, tratar formatos inconsistentes, lidar com valores ausentes e transformar variáveis categóricas em um formato numérico que o modelo possa entender.

As principais etapas de pré-processamento realizadas são:

1.  **Padronização dos nomes das colunas:** Converter nomes de colunas para minúsculas e remover caracteres especiais para facilitar o acesso.
2.  **Conversão de tipos:** Garantir que as colunas estejam nos tipos de dados corretos (ex: numérico para quantidades, string para nomes).
3.  **Diagnóstico inicial:** Inspecionar os dados após o carregamento para entender sua estrutura e identificar problemas iniciais.
4.  **Remoção de valores ausentes:** Lidar com dados faltantes em colunas essenciais para a análise.
5.  **Agrupamento por dimensões relevantes:** Agregar os dados pela granularidade desejada para a previsão (ano, filial, local, tipo, categoria, subcategoria).
6.  **Codificação de variáveis categóricas:** Transformar as colunas categóricas agregadas em representações numéricas (One-Hot Encoding).
7.  **Salvamento do dataset pré-processado:** Salvar o resultado do pré-processamento para uso nas etapas seguintes.

## 3. Escolha e Justificativa do Modelo (XGBoost)

Esta seção apresenta o modelo de Machine Learning selecionado para a previsão da quantidade movimentada de estoque e as razões técnicas para essa escolha.

### 3.1 Justificativa da Escolha do Modelo XGBoost para Previsão de Movimentação de Estoque

O presente projeto tem como objetivo desenvolver um modelo preditivo capaz de estimar a quantidade movimentada de produtos (entradas e saídas) nos próximos cinco anos, considerando as seguintes variáveis:

*   Ano da movimentação
*   Filial
*   Local de estoque
*   Tipo de movimentação
*   Categoria do produto
*   Subcategoria do produto

A previsão será utilizada para otimizar o planejamento logístico, reduzir custos operacionais e melhorar a acurácia na gestão de estoque em todas as unidades da empresa.

Após análise técnica e testes preliminares, foi definido que o modelo XGBoost (Extreme Gradient Boosting) será utilizado como base para a construção do algoritmo preditivo. A escolha se fundamenta nos seguintes critérios:

*   **Alta performance em dados tabulares:** O XGBoost é amplamente reconhecido por sua eficácia em problemas de regressão com dados estruturados.
*   **Eficiência computacional:** O algoritmo é otimizado para velocidade e uso de memória, ideal para grandes volumes de dados históricos.
*   **Capacidade de generalização:** Com mecanismos internos de regularização, o XGBoost ajuda a evitar overfitting.
*   **Flexibilidade na engenharia de variáveis:** O modelo se adapta bem a variáveis categóricas codificadas e permite incorporar variáveis temporais.

Mais detalhes sobre a escolha e referências técnicas podem ser encontrados na documentação original do notebook.

In [ ]:
referencias = dataset[[
    'ano_movimentacao',
    'filial',
    'nomelocalestoque',
    'subcategoria_produto',
    'categoria_produto',
    'tipo_movimentacao',
    'qte_movimentacao'
]].drop_duplicates()

In [ ]:
# Separar features (X) e target (y) do dataset_encoded
# A variável alvo agora é 'qte_movimentacao_transformed'
X_agregado = dataset_encoded.drop('qte_movimentacao_transformed', axis=1)
y_agregado = dataset_encoded['qte_movimentacao_transformed'] # Selecting as a Series

print("✅ Features e target separados no dataset agregado.")
display(X_agregado.head())
display(y_agregado.head())

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np # Import numpy

# Dividir os dados agregados em treino e teste
X_train_agregado, X_test_agregado, y_train_agregado, y_test_agregado = train_test_split(
    X_agregado,
    y_agregado.iloc[:, 0], # Explicitly select the first column
    test_size=0.3, # Usar 30% dos dados para teste
    random_state=42 # Para reprodutibilidade
)

# Convert boolean columns to integers (0 or 1)
for col in X_train_agregado.select_dtypes(include='bool').columns:
    X_train_agregado[col] = X_train_agregado[col].astype(int)

for col in X_test_agregado.select_dtypes(include='bool').columns:
    X_test_agregado[col] = X_test_agregado[col].astype(int)


print("✅ Dados agregados divididos em treino e teste e colunas booleanas convertidas para inteiros.")
print(f"Tamanho do conjunto de treino (features): {X_train_agregado.shape}")
print(f"Tamanho do conjunto de teste (features): {X_test_agregado.shape}")
print(f"Tamanho do conjunto de treino (target): {y_train_agregado.shape}")
print(f"Tamanho do conjunto de teste (target): {y_test_agregado.shape}")

# Optional: Re-check data types after conversion
# print("\nChecking data types of X_test_agregado after conversion:")
# print(X_test_agregado.dtypes)

In [ ]:
from xgboost import XGBRegressor
import numpy as np # Import numpy

# Instanciar e treinar o modelo XGBoost
modelo_agregado = XGBRegressor(n_estimators=100, max_depth=10, learning_rate=0.1, n_jobs=-1, random_state=42)

# Convert X_train_agregado to a dense NumPy array (if it's not already)
X_train_agregado_dense = X_train_agregado.toarray() if hasattr(X_train_agregado, 'toarray') else X_train_agregado

# Ensure y_train_agregado is a 1D array
y_train_agregado_np = y_train_agregado.values if hasattr(y_train_agregado, 'values') else y_train_agregado
y_train_agregado_np = y_train_agregado_np.flatten() if y_train_agregado_np.ndim > 1 else y_train_agregado_np


modelo_agregado.fit(X_train_agregado_dense, y_train_agregado_np)

print("✅ Modelo XGBoost treinado com dados agregados.")

## 4. Avaliação dos Resultados do Modelo Agregado

Após o treinamento do modelo XGBoost com os dados agregados e a variável alvo transformada, as seguintes métricas de avaliação foram obtidas no conjunto de teste:

* **MAE (Erro Absoluto Médio):** 1.22
* **RMSE (Raiz do Erro Quadrático Médio):** 1.58
* **R² (Coeficiente de Determinação):** 0.8568

### Discussão dos Resultados Iniciais

As métricas apresentadas oferecem uma visão inicial do desempenho do modelo na previsão da quantidade movimentada de estoque nos dados agregados:

* **R² de 0.8568:** Este valor indica que o modelo é capaz de explicar aproximadamente 85.7% da variabilidade na variável alvo transformada ('qte_movimentacao_transformed'). Um valor de R² próximo a 1 sugere que o modelo tem um bom ajuste aos dados de teste, capturando uma porção significativa da relação entre as features e a quantidade movimentada agregada.

* **MAE de 1.22:** O Erro Absoluto Médio representa a média das diferenças absolutas entre os valores reais e os valores previstos na variável alvo transformada. Um MAE de 1.22 significa que, em média, as previsões do modelo para a quantidade movimentada transformada estão a aproximadamente 1.22 unidades do valor real transformado. É importante lembrar que este erro está na escala logarítmica da variável alvo, e a interpretação direta em termos de quantidade real movimentada requer a aplicação da transformação inversa (exponencial).

* **RMSE de 1.58:** A Raiz do Erro Quadrático Médio é outra métrica que mede a magnitude dos erros de previsão. Ao contrário do MAE, o RMSE penaliza erros maiores de forma mais significativa devido ao quadrado das diferenças. Um RMSE de 1.58 sugere que os erros de previsão tendem a ser um pouco maiores do que o indicado pelo MAE, o que pode ser influenciado pela presença de alguns erros de previsão mais substanciais. Similar ao MAE, este valor está na escala logarítmica.

### Análise de Cada Métrica

* **R²:** O alto valor de R² é um indicativo positivo de que as features selecionadas e o modelo XGBoost são relevantes para prever a quantidade movimentada agregada. No entanto, é crucial não depender apenas do R², pois ele pode não refletir a performance real em casos com outliers ou distribuições de dados específicas.

* **MAE e RMSE:** A comparação entre MAE e RMSE pode fornecer insights sobre a distribuição dos erros. Quando o RMSE é significativamente maior que o MAE, isso sugere que o modelo comete alguns erros de previsão grandes. Neste caso, o RMSE (1.58) é ligeiramente maior que o MAE (1.22), indicando que pode haver alguns pontos no conjunto de teste onde o modelo teve dificuldades em fazer previsões precisas para a variável alvo transformada.

### Próximos Passos

A partir desta avaliação inicial, os próximos passos podem incluir:

* **Análise de Erros:** Investigar os casos com os maiores erros para entender as características dos dados onde o modelo tem mais dificuldade. Isso pode envolver analisar as distribuições de erros por categoria, filial, etc. (como já começado nas células seguintes).
* **Interpretação do Modelo:** Utilizar técnicas como SHAP (já iniciada nas células seguintes) para entender quais features são mais importantes para as previsões do modelo e como elas influenciam a saída do modelo transformado.
* **Considerar a Transformação Inversa:** Lembrar que as métricas estão na escala transformada. Para entender o desempenho do modelo em termos da quantidade real movimentada, seria necessário aplicar a transformação inversa às previsões e calcular as métricas na escala original. Isso é crucial para uma interpretação de negócio acurada.
* **Refinamento do Modelo:** Com base na análise de erros e na interpretação do modelo, pode ser necessário refinar os hiperparâmetros (já feito com Optuna) ou considerar outras abordagens de modelagem se o desempenho não for satisfatório para o problema de negócio.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Fazer previsões no conjunto de teste
y_pred_agregado = modelo_agregado.predict(X_test_agregado)

# Ensure predictions are a 1D array with the correct number of samples
y_pred_agregado = y_pred_agregado.reshape(-1)


# Calcular as métricas de regressão
mae_agregado = mean_absolute_error(y_test_agregado, y_pred_agregado)
mse_agregado = mean_squared_error(y_test_agregado, y_pred_agregado)
rmse_agregado = np.sqrt(mse_agregado)
r2_agregado = r2_score(y_test_agregado, y_pred_agregado)

# Exibir os resultados
print("📈 Métricas de Avaliação do Modelo (Dados Agregados):")
print(f"  MAE (Erro Absoluto Médio): {mae_agregado:.2f}")
print(f"  RMSE (Raiz do Erro Quadrático Médio): {rmse_agregado:.2f}")
print(f"  R² (Coeficiente de Determinação): {r2_agregado:.4f}")

## Análise de Erros, Causas e Proposição de Ajustes

Esta seção aprofunda a análise do desempenho do modelo, focando nos erros de previsão observados no conjunto de teste agregado. Entender onde e por que o modelo comete erros é crucial para identificar áreas de melhoria e refinar a abordagem de modelagem.

### Análise dos Erros Observados

As células de código a seguir calculam e analisam a distribuição dos erros absolutos do modelo XGBoost treinado com dados agregados. O foco está em entender como o erro varia em relação às features originais (Filial, Local de Estoque, Categoria do Produto, Subcategoria do Produto e Tipo de Movimentação).

Serão gerados DataFrames contendo as métricas de erro (média, desvio padrão e contagem) agrupadas por cada uma dessas features, permitindo identificar quais categorias ou combinações de categorias apresentam maior dificuldade de previsão para o modelo.

### Possíveis Causas dos Erros

Com base na análise da distribuição dos erros, algumas causas potenciais para as imprecisões do modelo podem incluir:

* **Variabilidade Intrínseca em Certas Categorias:** Algumas filiais, locais de estoque ou categorias de produtos podem ter padrões de movimentação de estoque inerentemente mais voláteis ou imprevisíveis, tornando a previsão mais desafiadora.
* **Dados Insuficientes para Algumas Combinações de Categorias:** Algumas combinações específicas de filial, local, tipo, categoria e subcategoria podem ter um número limitado de registros históricos no dataset agregado. Modelos de Machine Learning geralmente têm dificuldade em aprender padrões precisos com poucos exemplos.
* **Outliers e Eventos Raros:** A presença de movimentações de estoque atipicamente grandes ou pequenas (outliers) que não são representativas dos padrões gerais pode influenciar negativamente o treinamento do modelo e levar a erros significativos para esses casos ou casos semelhantes.
* **Limitações das Features Atuais:** As features utilizadas (ano, filial, local, tipo, categoria, subcategoria) podem não capturar toda a complexidade dos fatores que influenciam a movimentação de estoque. Fatores externos (eventos promocionais, sazonalidade não explicitamente modelada, mudanças na demanda, etc.) podem ser causas de erro.
* **Impacto da Transformação da Variável Alvo:** Embora a transformação logarítmica ajude a lidar com a assimetria, ela também pode complicar a interpretação direta dos erros e a capacidade do modelo de prever valores na escala original.
* **Movimentações Relacionadas a Contratos:** Algumas entradas e saídas de estoque podem estar ligadas a contratos específicos em vez de movimentações comuns. Isso pode gerar uma dificuldade para o modelo diferenciar esses tipos de movimentação, levando a discrepâncias nos valores previstos, especialmente se os volumes contratuais forem significativamente diferentes dos volumes de movimentação regular.

### Proposição de Ajustes Necessários

Considerando a análise de erros e suas possíveis causas, os seguintes ajustes e próximos passos são propostos:

* **Investigação Focada nas Áreas de Maior Erro:** Analisar em detalhe os subconjuntos de dados com os maiores erros médios (identificados nos agrupamentos por features) para entender as características específicas desses casos.
* **Coleta de Dados Adicionais:** Se a análise revelar que a falta de dados para certas combinações é um fator significativo, explorar a possibilidade de incorporar dados históricos adicionais, se disponíveis.
* **Tratamento Específico de Outliers:** Implementar estratégias mais robustas para identificar e lidar com outliers durante o pré-processamento, como winsorização ou remoção (com cautela).
* **Engenharia de Features Adicionais:** Explorar a criação de novas features que possam capturar melhor a sazonalidade (se aplicável), tendências temporais, ou interações entre as categorias que possam estar impactando a precisão. Considerar se informações sobre a natureza da movimentação (contratual vs. comum) podem ser incluídas como feature, caso estejam disponíveis no dataset original ou possam ser derivadas.
* **Experimentação com Outras Transformações ou Modelos:** Considerar outras transformações para a variável alvo ou experimentar com modelos de Machine Learning alternativos que possam ser mais robustos a outliers ou melhor capturar padrões complexos.
* **Análise dos Erros na Escala Original:** Conforme mencionado na seção anterior, é fundamental reverter as previsões para a escala original (`qte_movimentacao`) e analisar as métricas de erro (MAE, RMSE, MAPE, etc.) nessa escala para obter uma compreensão clara do desempenho do modelo em termos de negócio. A célula seguinte calculará métricas adicionais, incluindo MAPE e MedAE, que podem fornecer insights complementares.

In [ ]:
import numpy as np

# 1. Calculate errors for the full test set
# Create the DataFrame with the same index as y_test_agregado
df_erros_full_test = pd.DataFrame({
    'real': np.asarray(y_test_agregado)[:, 0] if hasattr(y_test_agregado, 'shape') and y_test_agregado.shape == (1101, 2) else np.asarray(y_test_agregado),
    'previsto': np.asarray(y_pred_agregado)[:, 0] if hasattr(y_pred_agregado, 'shape') and y_pred_agregado.shape == (1101, 2) else np.asarray(y_pred_agregado)
}, index=y_test_agregado.index) # Set the index explicitly


df_erros_full_test['erro'] = df_erros_full_test['real'] - df_erros_full_test['previsto']
df_erros_full_test['erro_absoluto'] = df_erros_full_test['erro'].abs()

# 2. Select the original categorical columns from the original aggregated dataset
# Use the index of y_test_agregado to select rows from the original aggregated dataset
original_categorical_cols = dataset_agrupado.loc[y_test_agregado.index, ['ano_movimentacao', 'filial', 'nomelocalestoque', 'tipo_movimentacao', 'categoria_produto', 'subcategoria_produto', 'qte_movimentacao']].copy()


# 3. Merge the error information with the original categorical columns
# The merge is done on the index, as df_erros_full_test and original_categorical_cols share the same index
df_erros_with_context = df_erros_full_test.merge(
    original_categorical_cols,
    left_index=True,
    right_index=True,
    how='left'
)

print("✅ DataFrame with errors and original categorical context created.")
display(df_erros_with_context.head(10))

In [ ]:
import numpy as np

# 1. Calculate errors for the aggregated test set
# Create the DataFrame with the same index as y_test_agregado
df_erros_agregado = pd.DataFrame({
    'real_agregado': np.asarray(y_test_agregado)[:, 0] if hasattr(y_test_agregado, 'shape') and y_test_agregado.shape == (1101, 2) else np.asarray(y_test_agregado),
    'previsto_agregado': np.asarray(y_pred_agregado)[:, 0] if hasattr(y_pred_agregado, 'shape') and y_pred_agregado.shape == (1101, 2) else np.asarray(y_pred_agregado)
}, index=y_test_agregado.index) # Set the index explicitly

df_erros_agregado['erro_agregado'] = df_erros_agregado['real_agregado'] - df_erros_agregado['previsto_agregado']
df_erros_agregado['erro_absoluto_agregado'] = df_erros_agregado['erro_agregado'].abs()

print("✅ DataFrame com erros do modelo agregado criado.")
display(df_erros_agregado.head())

In [ ]:
# Merge the aggregated errors with the aggregated test features
df_erros_agregado_with_context = df_erros_agregado.merge(
    X_test_agregado,
    left_index=True,
    right_index=True,
    how='left'
)

# List of categorical features (now encoded in X_test_agregado) to analyze errors by
# We need to identify the original categorical features from the encoded column names
# For this aggregated model, the original features are 'filial', 'nomelocalestoque',
# 'tipo_movimentacao', 'categoria_produto', 'subcategoria_produto'
# We will analyze the error distribution across the original categories by grouping
# the merged dataframe by the original categorical features from the aggregated dataset.

# Note: Since X_test_agregado contains one-hot encoded columns, direct grouping by
# the original categorical names is not straightforward without decoding or
# referencing the original aggregated dataset structure.

# Let's group by the relevant original categorical features from the aggregated dataset
# We can use the index of X_test_agregado to join with the original aggregated dataset
# to get the original categorical values.

# Alternatively, we can analyze error distribution by the one-hot encoded features
# directly, or perform grouping using the original dataset_agrupado indices.

# Given the previous steps, let's go back to the original dataset_agrupado
# and merge the aggregated errors based on index to get the original categorical columns.

df_erros_agregado_with_original_context = df_erros_agregado.merge(
    dataset_agrupado[grouping_columns], # Use the grouping columns from dataset_agrupado
    left_index=True,
    right_index=True,
    how='left'
)

print("✅ DataFrame com erros do modelo agregado e contexto original criado.")
display(df_erros_agregado_with_original_context.head())

# Now, analyze Mean Absolute Error by the original categorical features
categorical_features_for_agg_analysis = [
    'tipo_movimentacao',
    'filial',
    'nomelocalestoque',
    'categoria_produto',
    'subcategoria_produto'
]

for feature in categorical_features_for_agg_analysis:
    print(f"\n📈 Mean Absolute Error by {feature} (Aggregated Model):")
    erro_por_feature_agregado = df_erros_agregado_with_original_context.groupby(feature)['erro_absoluto_agregado'].agg(['mean', 'std', 'count']).sort_values(by='mean', ascending=False)
    display(erro_por_feature_agregado.head())

    # Optional: Create boxplots for each categorical feature against absolute error
    # Requires careful handling of potential large number of categories
    plt.figure(figsize=(12, 6))
    sns.boxplot(x=feature, y='erro_absoluto_agregado', data=df_erros_agregado_with_original_context)
    plt.title(f'Distribuição do Erro Absoluto por {feature} (Modelo Agregado)')
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

### Features Mais Importantes e Análise

Com base na análise SHAP (conforme visualizado no summary plot e nos dependence plots), as features mais importantes para o modelo XGBoost agregado na previsão da quantidade movimentada de estoque são:

1.  **tipo_movimentacao_SAIDA:** Esta é a feature mais importante, indicando o tipo de movimentação ser 'SAIDA'.
2.  **nomelocalestoque específico:** Esta one-hot encoding para um local de estoque específico é a segunda feature mais importante.
3.  **subcategoria_produto_:** A one-hot encoding para a subcategoria de produto ' ' é a terceira feature mais importante.
4.  **filial:** A one-hot encoding para a filial ' ' é a quarta feature mais importante.
5.  **categoria_produto:** A one-hot encoding para a categoria de produto ' ' é a quinta feature mais importante.

*(Note: As features listadas aqui refletem as one-hot encodings das variáveis categóricas. A importância é atribuída à presença de uma categoria específica).*

### Possíveis Motivos para a Importância dessas Features

*   **Tipo de Movimentação (SAIDA):** É natural que o tipo de movimentação (entrada vs. saída) seja a feature mais importante. A quantidade movimentada em uma saída tende a ter uma distribuição e magnitude diferentes de uma entrada, refletindo processos logísticos distintos (ex: consumo/distribuição vs. recebimento/armazenamento). O modelo capturou essa diferença fundamental.
*   **Local de Estoque e Filial Específicos:** A alta importância de 'confidencial' sugere que esses locais/filiais podem ter volumes de movimentação de estoque significativamente diferentes ou padrões de movimentação distintos em comparação com outros locais ou filiais. Isso pode ser devido ao volume geral de operações, tipos de produtos movimentados, ou processos logísticos específicos desses locais.
*   **Categoria e Subcategoria de Produto:** A importância da categoria e subcategoria 'confidencial' indica que produtos de 'confidencial' podem ter padrões de movimentação únicos. Isso pode ser impulsionado por regulamentações de segurança, demanda sazional (dependendo da atividade), ou características de consumo/reposição que os diferenciam de outras categorias de produtos. A alta importância tanto da categoria quanto da subcategoria 'EPI' sugere que essa granularidade específica de produto é um fator chave na previsão.

### Implicações para o Treinamento do Modelo

A importância dessas features tem várias implicações para o treinamento e o refinamento do modelo:

*   **Validação das Features:** A análise SHAP confirma que as features categóricas selecionadas (Filial, Local de Estoque, Tipo de Movimentação, Categoria, Subcategoria) são altamente relevantes para a previsão da quantidade movimentada agregada.
*   **Foco na Análise de Erros:** Ao investigar os erros do modelo, é particularmente útil analisar o desempenho em fatias de dados definidas por essas features importantes (ex: erros para movimentações de SAIDA, erros no 'ALMOXARIFADO CENTRAL - MÃO DE OBRA', erros para produtos 'EPI'). Isso pode ajudar a identificar se o modelo tem dificuldades específicas com certos tipos de movimentação, locais ou produtos.
*   **Potencial para Engenharia de Features Adicionais:** Embora essas features sejam importantes, a análise de erros pode revelar a necessidade de features adicionais que capturem nuances não explicadas por elas. Por exemplo, se 'EPI' tem alta importância, talvez adicionar features relacionadas a regulamentações de segurança ou sazonalidade específica de EPIs possa melhorar a precisão. Similarmente, se um local ou filial específico é importante, talvez dados sobre o tamanho do local, a equipe operacional ou o tipo de projetos suportados por essa filial/local possam ser úteis.
*   **Relevância do Agrupamento:** A importância das features categóricas no modelo agregado valida a estratégia de agrupar os dados por essas dimensões. As previsões geradas a essa granularidade parecem capturar as principais variações nos padrões de movimentação.
*   **Considerar a Transformação Inversa:** A análise SHAP foi realizada na variável alvo transformada (`qte_movimentacao_transformed`). Ao interpretar o impacto das features, devemos lembrar que ele está na escala transformada. Para entender o impacto na quantidade real movimentada, seria necessário considerar o efeito da transformação inversa.

In [ ]:
import shap
import matplotlib.pyplot as plt # Import matplotlib
import numpy as np # Import numpy

# Calculate SHAP values for the aggregated test set
# Use shap.TreeExplainer with the model's predict function and background data

# Use a sample of the training data as background data for the explainer
# Ensure X_train_agregado is a dense NumPy array for background data
X_train_agregado_dense = X_train_agregado.toarray() if hasattr(X_train_agregado, 'toarray') else X_train_agregado
background_data = shap.maskers.Independent(X_train_agregado_dense)


# Define the model's predict function
# Ensure X_test_agregado is a dense NumPy array for prediction
X_test_agregado_dense = X_test_agregado.toarray() if hasattr(X_test_agregado, 'toarray') else X_test_agregado


model_predict_function = lambda x: modelo_agregado.predict(x)


# Initialize the explainer, specifying the output names to indicate a single output
explainer_agregado = shap.Explainer(model_predict_function, background_data, output_names=['prediction'])

shap_values_agregado = explainer_agregado(X_test_agregado_dense)

# Generate the SHAP summary plot for the aggregated model
print("Generating SHAP summary plot for the aggregated model...")
plt.figure(figsize=(10, 15)) # Adjust figure size if needed
shap.summary_plot(shap_values_agregado, X_test_agregado_dense, show=False) # Use dense array for plotting
plt.tight_layout()
plt.show()

In [ ]:
# Identify the top features from the aggregated SHAP values
# We can use the mean absolute SHAP values to rank features
mean_abs_shap_values_agregado = abs(shap_values_agregado.values).mean(0)

# Get the names of the features from X_test_agregado
feature_names_agregado = X_test_agregado.columns

# Create a pandas Series for easier sorting and viewing
# Ensure mean_abs_shap_values_agregado is 1D
if mean_abs_shap_values_agregado.ndim == 2:
     mean_abs_shap_values_agregado = mean_abs_shap_values_agregado[:, 0]


feature_importance_agregado = pd.Series(mean_abs_shap_values_agregado, index=feature_names_agregado)

# Sort the features by importance in descending order and select the top N (e.g., top 5)
top_n = 5 # You can change this number
most_important_features_agregado = feature_importance_agregado.sort_values(ascending=False).head(top_n)

print(f"As top {top_n} features mais importantes para o modelo agregado:")
display(most_important_features_agregado)

# Generate dependence plots for the top N most important features
print(f"\nGenerating dependence plots for the top {top_n} features...")

# Attempt to handle the unexpected shape of shap_values_agregado.values for plotting
shap_values_for_plotting = shap_values_agregado.values

# Check if the shape is (1115, 2) and select the first column if so
if shap_values_for_plotting.shape == (1115, 2):
    print("Warning: SHAP values have unexpected shape (1115, 2). Attempting to plot using the first column.")
    # This selection is a heuristic based on the error message shape
    # It assumes the first column of this shape is relevant for plotting per-feature dependence
    shap_values_for_plotting = shap_values_for_plotting[:, 0]
elif shap_values_for_plotting.ndim == 3:
     # If it's 3D (observations, features, outputs), select the first output
     print("Warning: SHAP values are 3D. Attempting to plot for the first output.")
     shap_values_for_plotting = shap_values_for_plotting[:, :, 0]
elif shap_values_for_plotting.shape != X_test_agregado.shape:
     print(f"Warning: SHAP values shape {shap_values_for_plotting.shape} does not match X_test_agregado shape {X_test_agregado.shape}. Plotting might fail.")


# Ensure X_test_agregado is a NumPy array for consistent indexing
X_test_agregado_np = X_test_agregado.values if hasattr(X_test_agregado, 'values') else X_test_agregado


for feature_name in most_important_features_agregado.index:
    print(f"Generating dependence plot for: {feature_name}")
    # Pass the potentially reshaped SHAP values and the feature values as NumPy array
    shap.dependence_plot(
        feature_name,
        shap_values_for_plotting, # Use the potentially reshaped SHAP values
        X_test_agregado_np,      # Use X_test_agregado as NumPy array
        feature_names=feature_names_agregado.tolist(), # Explicitly pass feature names as a list
        show=False
    )
    plt.title(f'SHAP Dependence Plot for {feature_name}') # Add a title to the plot
    plt.tight_layout()
    plt.show()

### Métricas Adicionais de Avaliação e Implicações

Além das métricas padrão (MAE, RMSE, R²), calculamos o Erro Percentual Absoluto Médio (MAPE) e o Erro Mediano Absoluto (MedAE) para obter uma compreensão mais completa do desempenho do modelo agregado, especialmente considerando a transformação logarítmica da variável alvo e a natureza dos dados de movimentação de estoque.

*   **MAPE (Erro Percentual Absoluto Médio):**
    *   **Valor Obtido:** 49.85%
    *   **Interpretação:** O MAPE expressa o erro médio como uma porcentagem dos valores reais. Um MAPE de 49.85% na escala transformada sugere que, em média, as previsões do modelo para a quantidade movimentada transformada desviam em quase 50% dos valores reais transformados. É crucial notar que este valor está na escala logarítmica e não pode ser interpretado diretamente como um erro percentual na quantidade real movimentada. O MAPE é sensível a valores reais próximos de zero (que na escala transformada correspondem a valores próximos de `log1p(0)=0`), onde pequenas diferenças absolutas podem resultar em grandes erros percentuais.
    *   **Implicações:** Um MAPE na escala transformada de quase 50% indica que, embora o R² seja alto, o modelo pode ter dificuldades em prever com precisão a magnitude das movimentações em alguns casos, especialmente onde os volumes reais (e, portanto, os valores transformados) são menores. Isso reforça a necessidade de analisar os erros na escala original após a transformação inversa.

*   **MedAE (Erro Mediano Absoluto):**
    *   **Valor Obtido:** 1.02
    *   **Interpretação:** O MedAE é o erro absoluto no ponto médio da distribuição dos erros absolutos. É uma métrica mais robusta a outliers em comparação com o MAE, pois não é influenciada por alguns poucos erros muito grandes. Um MedAE de 1.02 na escala transformada significa que pelo menos metade dos erros absolutos das previsões estão abaixo de 1.02 (na escala transformada).
    *   **Implicações:** A comparação entre o MAE (1.22) e o MedAE (1.02) sugere que a distribuição dos erros absolutos na escala transformada é um pouco assimétrica, com alguns erros maiores "puxando" a média para cima em relação à mediana. Isso corrobora a análise anterior do RMSE vs. MAE e indica a presença de alguns casos no conjunto de teste onde o modelo teve um erro de previsão na escala transformada maior do que o típico.

### Considerações Finais sobre a Performance do Modelo Agregado

As métricas adicionais fornecem uma visão mais matizada da performance do modelo:

*   O alto R² indica que o modelo capturou a maior parte da variabilidade nos dados agregados transformados, validando a escolha das features e a abordagem de agrupamento.
*   O MAE e RMSE na escala transformada são razoáveis, mas o MAPE e a diferença entre MAE e MedAE sugerem que o modelo ainda enfrenta desafios na precisão pontual para certas instâncias, possivelmente aquelas com volumes de movimentação menores ou padrões atípicos.
*   **A interpretação mais crítica do desempenho do modelo requer a aplicação da transformação inversa (`expm1`) às previsões e o cálculo das métricas na escala original (`qte_movimentacao`).** Isso revelará o erro médio e mediano em termos das quantidades reais de estoque, que é a métrica mais relevante para o contexto de negócio.



In [ ]:
from sklearn.metrics import mean_absolute_percentage_error, median_absolute_error
import numpy as np

# Ensure y_test_agregado and y_pred_agregado are 1D NumPy arrays of the correct size (1101)
# Use np.array() for conversion and reshape(-1) to ensure 1D
y_test_agregado_np_full = np.array(y_test_agregado).reshape(-1)
y_pred_agregado_np_full = np.array(y_pred_agregado).reshape(-1)

# Double check the shapes after explicit conversion
# print(f"Shape of y_test_agregado_np_full: {y_test_agregado_np_full.shape}")
# print(f"Shape of y_pred_agregado_np_full: {y_pred_agregado_np_full.shape}")

# Calculate MAPE only for non-zero actual values to avoid division by zero issues
# Create boolean mask from the full true values array
non_zero_mask = y_test_agregado_np_full != 0

# Apply the mask to both true and predicted values to get non-zero subsets of the same length
# Ensure the mask is applied to arrays of the same size
if y_test_agregado_np_full.shape == y_pred_agregado_np_full.shape:
    y_test_original_non_zero = y_test_agregado_np_full[non_zero_mask]
    y_pred_original_non_zero = y_pred_agregado_np_full[non_zero_mask]
else:
    # Handle the case where shapes still don't match after explicit conversion (should not happen with this fix)
    print("Error: Shapes of y_test_agregado_np_full and y_pred_agregado_np_full do not match.")
    y_test_original_non_zero = np.array([])
    y_pred_original_non_zero = np.array([])


# Calculate MAPE
mape_agregado = np.mean(np.abs((y_test_original_non_zero - y_pred_original_non_zero) / y_test_original_non_zero)) * 100 if len(y_test_original_non_zero) > 0 else np.nan


# Calculate Median Absolute Error for the initial model predictions
# Ensure full arrays are 1D for MedAE calculation (already done above)
medae_agregado = median_absolute_error(y_test_agregado_np_full, y_pred_agregado_np_full)


print("\n📈 Métricas Adicionais de Avaliação do Modelo Inicial (Dados Agregados):")
if not np.isnan(mape_agregado):
    print(f"  MAPE (Erro Percentual Absoluto Médio): {mape_agregado:.2f}%")
else:
    print("  MAPE (Erro Percentual Absoluto Médio): Não calculável (todos os valores reais são zero)")
print(f"  MedAE (Erro Mediano Absoluto): {medae_agregado:.2f}")

## 6. Ajuste de Hiperparâmetros com Optuna

A performance do nosso modelo XGBoost agregado, desenvolvido para prever a quantidade movimentada de estoque, é significativamente influenciada pelas suas configurações internas, conhecidas como hiperparâmetros. Diferente dos parâmetros que o modelo aprende diretamente dos dados (como os pesos em uma regressão), os hiperparâmetros são definidos antes do treinamento e controlam aspectos como a complexidade do modelo e a velocidade de aprendizado.

### Por Que Ajustar os Hiperparâmetros do Nosso Modelo?

Utilizar os hiperparâmetros padrão do XGBoost pode não ser o ideal para o nosso problema específico de previsão de movimentação de estoque. Cada dataset e tarefa de modelagem tem características únicas, e encontrar a combinação certa de hiperparâmetros é crucial para extrair o máximo potencial do modelo. O ajuste visa:

*   **Otimizar a Precisão:** Encontrar as configurações que minimizem o erro de previsão (medido pelo RMSE na escala transformada) no nosso dataset agregado.
*   **Melhorar a Capacidade de Generalização:** Garantir que o modelo não apenas se ajuste bem aos dados históricos de treino, mas também seja capaz de fazer previsões precisas para futuras movimentações de estoque que ele nunca viu. Isso é vital para a aplicação prática da previsão.
*   **Gerenciar a Complexidade:** Controlar a profundidade das árvores, a taxa de aprendizado e a regularização para evitar que o modelo se torne excessivamente complexo (overfitting), o que poderia levar a um bom desempenho no treino, mas ruim em novos dados.

### Optuna: Nosso Otimizador de Hiperparâmetros

Escolhemos a biblioteca Optuna para automatizar esse processo de busca pelos melhores hiperparâmetros. Optuna é eficiente e utiliza estratégias de busca inteligentes (como TPE) que aprendem com os resultados de tentativas anteriores para explorar as configurações mais promissoras.

O processo no nosso notebook funciona assim:

1.  **Definição da Função Objetivo:** Criamos uma função (`objective`) que o Optuna executa repetidamente. Para cada execução (chamada de "trial"), o Optuna sugere um conjunto diferente de hiperparâmetros.
2.  **Treinamento e Avaliação:** Dentro da função objetivo, treinamos um modelo XGBoost com os hiperparâmetros sugeridos e o avaliamos usando validação cruzada no nosso conjunto de treino agregado. A métrica que retornamos é o RMSE médio obtido nessa avaliação.
3.  **Otimização:** O Optuna executa múltiplos trials (definimos 30), buscando encontrar o conjunto de hiperparâmetros que resulte no menor RMSE médio possível.

### Configuração Específica do Ajuste para a Previsão de Estoque

No nosso caso, estamos otimizando uma série de hiperparâmetros que controlam aspectos cruciais do nosso modelo XGBoost:

*   **`n_estimators` (Número de Árvores):** Quantas árvores de decisão compõem nosso modelo ensemble. Mais árvores podem melhorar a performance, mas também aumentam o tempo de treino e o risco de overfitting se não controladas.
*   **`learning_rate` (Taxa de Aprendizado):** Controla o quanto cada nova árvore contribui para a previsão final. Valores menores exigem mais árvores, mas podem levar a um modelo mais robusto.
*   **`max_depth` (Profundidade Máxima da Árvore):** Define quão complexa cada árvore de decisão individual pode ser. Profundidades maiores permitem capturar interações mais complexas entre as features (o que pode ser relevante dada a alta dimensionalidade após a codificação one-hot), mas aumentam o risco de overfitting.
*   **`subsample` e `colsample_bytree`:** Controlam o subamostragem de dados e features para cada árvore. Isso ajuda a tornar o modelo mais robusto e a reduzir a correlação entre as árvores, combatendo o overfitting. Dado o volume e a variedade das nossas movimentações de estoque, essas configurações são importantes para garantir que o modelo não se especialize demais em subconjuntos específicos de dados ou features.
*   **`min_child_weight` e `gamma`:** Parâmetros que controlam as condições para que uma árvore continue a se dividir. Eles ajudam a controlar a complexidade da árvore e a evitar overfitting em dados ruidosos.
*   **`reg_alpha` e `reg_lambda` (Regularização L1/L2):** Termos que penalizam a complexidade do modelo, forçando os pesos das features a serem menores. Essenciais para evitar overfitting, especialmente em datasets com muitas features (como o nosso após a codificação one-hot).

### Validação Cruzada (K-Fold) e a Escolha de 3 Splits

Para avaliar cada conjunto de hiperparâmetros sugerido pelo Optuna, utilizamos validação cruzada K-Fold com 3 splits (`n_splits=3`).

*   **Por que K-Fold?** A movimentação de estoque pode ter padrões variados. Usar validação cruzada nos permite treinar e avaliar o modelo em diferentes subconjuntos do nosso dataset de treino agregado. Isso nos dá uma estimativa mais confiável do desempenho médio esperado do modelo em dados não vistos, em comparação com uma única divisão treino/validação. É como testar o modelo em 3 "versões" diferentes do nosso histórico de treino.
*   **Por que 3 Splits?** A escolha de 3 splits foi um equilíbrio pragmático. Um número maior de splits (como 5 ou 10) forneceria uma estimativa de performance ainda mais estável, mas cada trial do Optuna exigiria 5 ou 10 treinamentos e avaliações do modelo, aumentando consideravelmente o tempo total da otimização (que já pode ser longa com 30 trials em um dataset desse tamanho). 3 splits oferecem uma validação razoável sem tornar o processo proibitivamente demorado. A randomização (`shuffle=True`) garante que cada fold seja representativo do conjunto de treino geral.

### Análise dos Resultados da Otimização

Após a execução do Optuna, observaremos os "Melhores hiperparâmetros" encontrados e o "Melhor RMSE" na validação cruzada.

*   **Os Melhores Hiperparâmetros:** A combinação específica de valores encontrada pelo Optuna representa a configuração que, em média, obteve o melhor desempenho (menor RMSE) nos 3 folds de validação cruzada. Esses valores são a base para treinar nosso modelo final.
*   **O Melhor RMSE:** Este valor (aproximadamente 1.70) é a nossa melhor estimativa do erro quadrático médio que podemos esperar do modelo otimizado em dados não vistos, com base na nossa validação cruzada. Ele nos dá uma medida quantitativa da performance esperada na escala transformada. Comparado com o RMSE do modelo inicial (1.58), podemos avaliar o impacto da otimização. É interessante notar que o RMSE otimizado foi ligeiramente maior neste caso, o que pode sugerir que o modelo inicial já estava perto de uma configuração razoável, ou que mais trials/folds poderiam encontrar uma combinação ainda melhor.

### Validação da Performance e Próximos Passos

Os resultados da otimização com Optuna fornecem uma importante validação interna da performance do nosso modelo agregado. O "Melhor RMSE" da validação cruzada nos dá confiança na capacidade de generalização do modelo com os hiperparâmetros encontrados.

O próximo passo lógico é treinar o modelo final utilizando o conjunto completo de dados de treino agregados e os melhores hiperparâmetros identificados pelo Optuna. Em seguida, avaliaremos este modelo final no conjunto de teste (que o Optuna não viu durante a otimização) para obter uma medida final e independente do seu desempenho.

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
import optuna
import numpy as np # Import numpy

def objective(trial):
    """Defines the objective function for Optuna."""
    params = {
        "objective": "reg:squarederror",
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.05, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.05, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0, 0.4),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 1),
        "random_state": 42,
        "n_jobs": -1,
    }

    kf = KFold(n_splits=3, shuffle=True, random_state=42) # Using 3 folds for faster execution
    rmse_scores = []

    # Perform cross-validation on the training data
    for fold, (train_index, val_index) in enumerate(kf.split(X_train_agregado, y_train_agregado)):
        X_train_fold, X_val_fold = X_train_agregado.iloc[train_index], X_train_agregado.iloc[val_index]
        y_train_fold, y_val_fold = y_train_agregado.iloc[train_index], y_train_agregado.iloc[val_index]

        # Convert y_train_fold to a NumPy array
        y_train_fold_np = y_train_fold.values if hasattr(y_train_fold, 'values') else y_train_fold


        model = xgb.XGBRegressor(**params)
        model.fit(X_train_fold, y_train_fold_np) # Removed eval_set and early_stopping_rounds
        predictions = model.predict(X_val_fold)
        mse = mean_squared_error(y_val_fold, predictions) # Removed squared=False
        rmse = np.sqrt(mse) # Calculate RMSE manually
        rmse_scores.append(rmse)

    # Return the average RMSE across folds
    return sum(rmse_scores) / len(rmse_scores)

In [ ]:
!pip install optuna

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)

In [ ]:
print('Melhores hiperparâmetros:', study.best_params)
print('Melhor RMSE:', study.best_value)

In [ ]:
import matplotlib.pyplot as plt
optuna.visualization.plot_optimization_history(study)
plt.show()

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Treinar o modelo final com os melhores hiperparâmetros encontrados pelo Optuna
best_params = study.best_params
modelo_final_otimizado = xgb.XGBRegressor(**best_params, random_state=42, n_jobs=-1)

# Convert y_train_agregado to a NumPy array
y_train_agregado_np = y_train_agregado.values if hasattr(y_train_agregado, 'values') else y_train_agregado

modelo_final_otimizado.fit(X_train_agregado, y_train_agregado_np)

print("✅ Modelo XGBoost final treinado com hiperparâmetros otimizados.")

# Fazer previsões no conjunto de teste com o modelo otimizado
y_pred_otimizado = modelo_final_otimizado.predict(X_test_agregado)

# Calcular e exibir as métricas de regressão para o modelo otimizado
mae_otimizado = mean_absolute_error(y_test_agregado, y_pred_otimizado)
mse_otimizado = mean_squared_error(y_test_agregado, y_pred_otimizado)
rmse_otimizado = np.sqrt(mse_otimizado)
r2_otimizado = r2_score(y_test_agregado, y_pred_otimizado)

print("\n📈 Métricas de Avaliação do Modelo Otimizado (Dados Agregados):")
print(f"  MAE (Erro Absoluto Médio): {mae_otimizado:.2f}")
print(f"  RMSE (Raiz do Erro Quadrático Médio): {rmse_otimizado:.2f}")
print(f"  R² (Coeficiente de Determinação): {r2_otimizado:.4f}")

## 4. Rejustificativa do Problema e Transição para Clusterização

Apesar da capacidade do modelo XGBoost em capturar a variabilidade na escala transformada, a avaliação na escala original revelou desafios significativos na previsão acurada da magnitude das movimentações de estoque, especialmente para grandes volumes. O alto MAE, RMSE, e MAPE na escala original, juntamente com o baixo R², indicam que prever a quantidade exata com precisão suficiente para o planejamento logístico ótimo é complexo com a abordagem atual.

Diante disso, e considerando a necessidade de entender quais produtos/categorias, em quais locais/filiais, mais entram e saem, redefinimos o problema de negócio. O foco passa a ser **identificar e agrupar padrões de movimentação de estoque** para diferentes combinações de dimensões (Ano, Filial, Local de Estoque, Tipo de Movimentação, Categoria, Subcategoria).

Esta redefinição nos leva à abordagem de **Clusterização**. Ao agrupar as movimentações agregadas com base em suas características (como volume total e variabilidade), podemos segmentar o estoque em grupos com comportamentos distintos. Essa segmentação permitirá:

*   Desenvolver estratégias de gestão de estoque mais direcionadas e eficazes para cada grupo.
*   Priorizar a atenção e os recursos para os grupos de maior volume ou variabilidade.
*   Obter insights sobre os padrões de consumo e reposição em diferentes partes da operação.

Portanto, o restante deste notebook se concentrará na aplicação de técnicas de clusterização para alcançar essa nova meta de negócio.

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error

# 1. Apply the inverse transformation (expm1)
# Handle negative values by applying expm1 to the absolute value and then multiplying by -1
y_test_original = np.sign(y_test_agregado) * np.expm1(np.abs(y_test_agregado))
y_pred_original = np.sign(y_pred_otimizado) * np.expm1(np.abs(y_pred_otimizado))

# Ensure the inverse transformed values are non-negative where the original qte_movimentacao was non-negative (ENTRADA)
# This is a safeguard, as expm1 of positive values should be positive.
y_test_original[y_test_agregado >= 0] = np.expm1(y_test_agregado[y_test_agregado >= 0])
y_pred_original[y_pred_otimizado >= 0] = np.expm1(y_pred_otimizado[y_pred_otimizado >= 0])

# Also, ensure output is non-negative for ENTREDA type (where original value was >= 0)
# This might be redundant with the above, but adds an extra layer of correction based on original type if needed.
# This requires access to the original type_movimentacao. For now, we rely on the sign of the transformed value.

# Clip predictions to be non-negative for 'ENTRADA' types based on original data if necessary
# This would require merging back the original type_movimentacao to the test set predictions.
# For this step, let's assume the sign handling above is sufficient based on the transformed data.
# If original types are needed, we'd need to merge df_erros_agregado_with_original_context.

# Let's proceed with the metrics calculation assuming the sign handling on transformed values is the basis.

# 2. Calculate MAE, RMSE, and R² in the original scale
mae_original = mean_absolute_error(y_test_original, y_pred_original)
mse_original = mean_squared_error(y_test_original, y_pred_original)
rmse_original = np.sqrt(mse_original)
r2_original = r2_score(y_test_original, y_pred_original)

# 3. Calculate MAPE and MedAE in the original scale
# Calculate MAPE only for non-zero actual values to avoid division by zero issues
non_zero_mask_original = y_test_original != 0

y_test_original_non_zero = y_test_original[non_zero_mask_original]
y_pred_original_non_zero = y_pred_original[non_zero_mask_original]


mape_original = np.mean(np.abs((y_test_original_non_zero - y_pred_original_non_zero) / y_test_original_non_zero)) * 100 if len(y_test_original_non_zero) > 0 else np.nan

medae_original = median_absolute_error(y_test_original, y_pred_original)

# 4. Discuss the calculated metrics (will be done in the next markdown cell)
print("\n📈 Métricas de Avaliação do Modelo Otimizado (Escala Original):")
print(f"  MAE (Erro Absoluto Médio): {mae_original:.2f}")
print(f"  RMSE (Raiz do Erro Quadrático Médio): {rmse_original:.2f}")
print(f"  R² (Coeficiente de Determinação): {r2_original:.4f}")
if not np.isnan(mape_original):
    print(f"  MAPE (Erro Percentual Absoluto Médio): {mape_original:.2f}%")
else:
    print("  MAPE (Erro Percentual Absoluto Médio): Não calculável (todos os valores reais são zero)")
print(f"  MedAE (Erro Mediano Absoluto): {medae_original:.2f}")

In [ ]:
import numpy as np
import pandas as pd

# Re-calculate original scale metrics (already done in previous step, but good to have available)
# Apply the inverse transformation (expm1)
y_test_original = np.sign(y_test_agregado) * np.expm1(np.abs(y_test_agregado))
y_pred_original = np.sign(y_pred_otimizado) * np.expm1(np.abs(y_pred_otimizado))

# Ensure the inverse transformed values are non-negative where the original qte_movimentacao was non-negative (ENTRADA)
y_test_original[y_test_agregado >= 0] = np.expm1(y_test_agregado[y_test_agregado >= 0])
y_pred_original[y_pred_otimizado >= 0] = np.expm1(y_pred_otimizado[y_pred_otimizado >= 0])

# Calculate original scale metrics
mae_original = mean_absolute_error(y_test_original, y_pred_original)
rmse_original = np.sqrt(mean_squared_error(y_test_original, y_pred_original))
r2_original = r2_score(y_test_original, y_pred_original)

non_zero_mask_original = y_test_original != 0
y_test_original_non_zero = y_test_original[non_zero_mask_original]
y_pred_original_non_zero = y_pred_original[non_zero_mask_original]

mape_original = np.mean(np.abs((y_test_original_non_zero - y_pred_original_non_zero) / y_test_original_non_zero)) * 100 if len(y_test_original_non_zero) > 0 else np.nan
medae_original = median_absolute_error(y_test_original, y_pred_original)


# 1. Refer back to the error analysis in the transformed scale (using df_erros_agregado_with_original_context)
# This dataframe contains errors in the transformed scale along with original categorical features.
# We can merge the original scale errors for comparison.

# Create a DataFrame for original scale errors
df_erros_original = pd.DataFrame({
    'real_original': y_test_original,
    'previsto_original': y_pred_original
}, index=y_test_agregado.index)

df_erros_original['erro_original'] = df_erros_original['real_original'] - df_erros_original['previsto_original']
df_erros_original['erro_absoluto_original'] = df_erros_original['erro_original'].abs()

# Merge original scale errors with the context dataframe
df_erros_full_context = df_erros_agregado_with_original_context.merge(
    df_erros_original[['erro_original', 'erro_absoluto_original']],
    left_index=True,
    right_index=True,
    how='left'
)


# 2. Consider the SHAP feature importance analysis (most_important_features_agregado)
# This variable already contains the top features and their importance scores.


# 3. Relate original scale metrics to error analysis and SHAP values.
# Analyze Mean Absolute Error in original scale by the original categorical features
print("\n📈 Mean Absolute Error by feature (Original Scale):")
categorical_features_for_agg_analysis = [
    'tipo_movimentacao',
    'filial',
    'nomelocalestoque',
    'categoria_produto',
    'subcategoria_produto'
]

for feature in categorical_features_for_agg_analysis:
    print(f"\n  MAE by {feature} (Original Scale):")
    # Group by the original categorical feature and calculate mean/std/count of original absolute error
    erro_por_feature_original = df_erros_full_context.groupby(feature)['erro_absoluto_original'].agg(['mean', 'std', 'count']).sort_values(by='mean', ascending=False)
    display(erro_por_feature_original.head())


# Discuss the relationship between SHAP importance and original scale errors in the markdown cell.

# 4. Discuss the impact of transformation and inverse transformation.
# This will be done in the markdown cell.

# 5. Provide a comprehensive discussion in the markdown cell.

# Display the original scale metrics again for easy reference during discussion
print("\n📈 Métricas de Avaliação do Modelo Otimizado (Escala Original):")
print(f"  MAE (Erro Absoluto Médio): {mae_original:.2f}")
print(f"  RMSE (Raiz do Erro Quadrático Médio): {rmse_original:.2f}")
print(f"  R² (Coeficiente de Determinação): {r2_original:.4f}")
if not np.isnan(mape_original):
    print(f"  MAPE (Erro Percentual Absoluto Médio): {mape_original:.2f}%")
else:
    print("  MAPE (Erro Percentual Absoluto Médio): Não calculável (todos os valores reais são zero)")
print(f"  MedAE (Erro Mediano Absoluto): {medae_original:.2f}")


In [ ]:
!pip install python-calamine